In [12]:
# ---------- Imports ----------
import glob
import re
import os
import numpy as np
import pandas as pd
from datetime import date

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)

# ---------- Settings (same as the collection notebook) ----------
MODEL = "gfs_seamless"
CITIES = {  # city: (NOAA station id, lat, lon)
    "Kolkata": ("IN024141500", 22.650, 88.450),
    "Visakhapatnam": ("IN001200200", 17.717, 83.233),
    "Thiruvananthapuram": ("IN010100700", 8.467, 76.950),
}
TEMP_START, TEMP_END = "2022-06-01", "2025-08-31"
RAIN_START, RAIN_END = "2024-01-20", "2025-08-31"
LEADS = range(1, 8)
INTERIM, CLEAN = "interim", "clean"
os.makedirs(CLEAN, exist_ok=True)

# ---------- Load the stored tables (Parquet from the storage step) ----------
noaa_path = sorted(glob.glob(f"{INTERIM}/noaa_ghcnd_3stations_*.parquet"))[-1]
DATE_COLLECTED = re.search(r"\d{4}-\d{2}-\d{2}", noaa_path).group()   # date taken from the file itself

noaa = pd.read_parquet(noaa_path)
om = pd.read_parquet(f"{INTERIM}/openmeteo_{MODEL}_hourly_{DATE_COLLECTED}.parquet")
print("NOAA     :", noaa.shape)
print("Open-Meteo:", om.shape)
N_NOAA_RAW, N_OM_RAW = len(noaa), len(om)      # raw row counts for the cleaning log

# ---------- Cleaning log: one line per step ----------
log = []
def add_log(step, rows_before, rows_after, note):
    log.append({"step": step, "rows_before": rows_before,
                "rows_after": rows_after, "rows_affected": rows_before - rows_after,
                "note": note})
    print(f"logged: {step}")

# ---------- Quick look before any change ----------
print("\nNOAA dtypes:\n", noaa.dtypes.to_string())
print("\nNOAA elements:", noaa["element"].value_counts().head(6).to_dict())

NOAA     : (137187, 9)
Open-Meteo: (105192, 17)

NOAA dtypes:
 id                  object
date        datetime64[ns]
element             object
value                int64
m_flag              object
q_flag              object
s_flag              object
obs_time           float64
city                object

NOAA elements: {'PRCP': 54602, 'TAVG': 38454, 'TMAX': 24222, 'TMIN': 19909}


In [13]:
# ---------- Spellings and case: before ----------
print("city values before   :", sorted(noaa["city"].unique()))
print("element values before:", noaa["element"].nunique(), "different elements")

# ---------- Fix text: strip spaces, same case ----------
for df in (noaa, om):
    old = df["city"].copy()
    df["city"] = df["city"].str.strip().str.title()
    print("city values changed   :", int((old != df["city"]).sum()))
old = noaa["element"].copy()
noaa["element"] = noaa["element"].str.strip().str.upper()
print("element values changed:", int((old != noaa["element"]).sum()))
print("city values after     :", sorted(noaa["city"].unique()))

# ---------- Dates: already real dates (YYYYMMDD numbers fixed at storage) ----------
print("\ndate dtype:", noaa["date"].dtype, "| time dtype:", om["time"].dtype)

# ---------- Quality flags before any filter: flagged rows per city (all years) ----------
print("\nrows with a quality flag, all years:",
      noaa.groupby("city")["q_flag"].apply(lambda s: int(s.notna().sum())).to_dict())

# ---------- Dates: keep the project window only ----------
n0 = len(noaa)
noaa = noaa[(noaa["date"] >= TEMP_START) & (noaa["date"] <= TEMP_END)].copy()
add_log("NOAA: keep project window", n0, len(noaa), f"{TEMP_START} to {TEMP_END}")

n0 = len(om)
start = pd.Timestamp(TEMP_START) - pd.Timedelta(days=1)      # one extra day for the rain window
end = pd.Timestamp(TEMP_END) + pd.Timedelta(days=1)
om = om[(om["time"] >= start) & (om["time"] < end)].copy()
add_log("Open-Meteo: keep project window", n0, len(om), "one extra day before start")

# ---------- Quality flag: keep rows that passed the check ----------
print("\nrows with a quality flag inside the project window:", int(noaa["q_flag"].notna().sum()))
n0 = len(noaa)
noaa = noaa[noaa["q_flag"].isna()].copy()
add_log("NOAA: drop rows failing quality check", n0, len(noaa), "q_flag not blank")

# ---------- Keep the two elements we analyse ----------
n0 = len(noaa)
noaa = noaa[noaa["element"].isin(["TMAX", "PRCP"])].copy()
add_log("NOAA: keep TMAX and PRCP", n0, len(noaa), "other elements not used")

# ---------- Missing-value code -9999 ----------
n0 = len(noaa)
noaa = noaa[noaa["value"] != -9999].copy()
add_log("NOAA: drop -9999 codes", n0, len(noaa), "GHCN missing code")

# ---------- Units: tenths to deg C and mm ----------
noaa["value_clean"] = noaa["value"] / 10
noaa["unit"] = noaa["element"].map({"TMAX": "deg C", "PRCP": "mm"})
print("\nunits fixed (before = tenths, after = deg C / mm):")
print(noaa.groupby("element")[["value", "value_clean"]].agg(["min", "max"]).to_string())

city values before   : ['Kolkata', 'Thiruvananthapuram', 'Visakhapatnam']
element values before: 4 different elements
city values changed   : 0
city values changed   : 0
element values changed: 0
city values after     : ['Kolkata', 'Thiruvananthapuram', 'Visakhapatnam']

date dtype: datetime64[ns] | time dtype: datetime64[ns]

rows with a quality flag, all years: {'Kolkata': 5, 'Thiruvananthapuram': 1, 'Visakhapatnam': 7}
logged: NOAA: keep project window
logged: Open-Meteo: keep project window

rows with a quality flag inside the project window: 0
logged: NOAA: drop rows failing quality check
logged: NOAA: keep TMAX and PRCP
logged: NOAA: drop -9999 codes

units fixed (before = tenths, after = deg C / mm):
        value       value_clean       
          min   max         min    max
element                               
PRCP        0  2111         0.0  211.1
TMAX      190   446        19.0   44.6


In [14]:
import json

# ---------- Rule: station and model grid point are the same place if under 10 km ----------
MAX_KM = 10

def haversine(lat1, lon1, lat2, lon2):
    """Distance in km between two points on the earth."""
    p1, p2 = np.radians(lat1), np.radians(lat2)
    a = (np.sin((p2 - p1) / 2) ** 2
         + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lon2 - lon1) / 2) ** 2)
    return 2 * 6371 * np.arcsin(np.sqrt(a))

# ---------- Grid point from the Open-Meteo response, station from the inventory ----------
rows = []
for city, (sid, lat, lon) in CITIES.items():
    with open(f"raw/openmeteo_{MODEL}_{city}_2024_{DATE_COLLECTED}.json", encoding="utf-8") as f:
        js = json.load(f)
    d = haversine(lat, lon, js["latitude"], js["longitude"])
    rows.append({"city": city, "station": sid, "station_lat": lat, "station_lon": lon,
                 "grid_lat": round(js["latitude"], 3), "grid_lon": round(js["longitude"], 3),
                 "distance_km": round(d, 1), "same_place": d < MAX_KM})
match = pd.DataFrame(rows)
print(match.to_string(index=False))
print("\nall three matched:", bool(match["same_place"].all()))

# ---------- One station per city, so no second NOAA source to merge ----------
print("\nstations per city:")
print(noaa.groupby("city")["id"].nunique().to_string())

# ---------- Duplicates inside each source: exact rows, then key ----------
for name, df, key in [("NOAA", noaa, ["city", "date", "element"]),
                      ("Open-Meteo", om, ["city", "time"])]:
    n0 = len(df)
    exact = int(df.duplicated().sum())
    by_key = int(df.duplicated(key).sum())
    print(f"\n{name}: exact duplicates {exact}, duplicate key {by_key}")
    df.drop_duplicates(inplace=True)
    add_log(f"{name}: drop exact duplicates", n0, len(df), "rule: all columns equal")
    n0 = len(df)
    df.drop_duplicates(key, inplace=True)
    add_log(f"{name}: drop duplicate key", n0, len(df), "rule: key " + "+".join(key))

# ---------- Log so far ----------
print()
print(pd.DataFrame(log)[["step", "rows_before", "rows_after", "rows_affected"]].to_string(index=False))

              city     station  station_lat  station_lon  grid_lat  grid_lon  distance_km  same_place
           Kolkata IN024141500       22.650       88.450    22.668    88.477          3.4        True
     Visakhapatnam IN001200200       17.717       83.233    17.748    83.203          4.7        True
Thiruvananthapuram IN010100700        8.467       76.950     8.493    76.992          5.5        True

all three matched: True

stations per city:
city
Kolkata               1
Thiruvananthapuram    1
Visakhapatnam         1

NOAA: exact duplicates 0, duplicate key 0
logged: NOAA: drop exact duplicates
logged: NOAA: drop duplicate key

Open-Meteo: exact duplicates 0, duplicate key 0
logged: Open-Meteo: drop exact duplicates
logged: Open-Meteo: drop duplicate key

                                 step  rows_before  rows_after  rows_affected
            NOAA: keep project window       137187       10385         126802
      Open-Meteo: keep project window       105192       85608         

In [15]:
# ---------- Log: remove repeated lines if a block was run twice ----------
log = pd.DataFrame(log).drop_duplicates().to_dict("records")

# ---------- Station side: full calendar, one row per city and date ----------
obs = noaa.pivot_table(index=["city", "date"], columns="element",
                       values="value_clean").reset_index()
days = pd.date_range(TEMP_START, TEMP_END)
calendar = pd.MultiIndex.from_product([list(CITIES), days],
                                      names=["city", "date"]).to_frame(index=False)
present = len(obs)
obs = calendar.merge(obs, on=["city", "date"], how="left")   # missing days appear as NaN
log.append({"step": "Rebuild full calendar", "rows_before": present, "rows_after": len(calendar),
            "rows_affected": len(calendar) - present, "note": "gap days added as NaN"})
print("station rows:", present, "-> calendar rows:", len(calendar))

# ---------- Forecast side: daily TMAX and daily rain per lead ----------
temp_parts, rain_parts = [], []
for city, g in om.groupby("city"):
    g = g.set_index("time")
    gw = g.copy()
    gw.index = gw.index + pd.Timedelta(hours=15)   # hour stamped 09:00 on day D-1 becomes 00:00 on day D
    t = pd.DataFrame({"city": city, "date": days})
    r = pd.DataFrame({"city": city, "date": days})
    for k in LEADS:
        # ---------- TMAX: highest hour of the day, only if all 24 hours exist ----------
        col = f"temperature_2m_previous_day{k}"
        mx = g[col].resample("D").max()
        n = g[col].resample("D").count()
        t[f"fc_tmax_lead{k}"] = mx.where(n == 24).reindex(days).values
        # ---------- Rain: sum over 09:00 (D-1) to 08:00 (D) stamps, all 24 hours needed ----------
        pcol = f"precipitation_previous_day{k}"
        r[f"fc_rain_lead{k}"] = gw[pcol].resample("D").sum(min_count=24).reindex(days).values
    # ---------- Same lead 1 rain by calendar day, only to check the alignment ----------
    r["fc_rain_calday_lead1"] = g["precipitation_previous_day1"].resample("D").sum(min_count=24).reindex(days).values
    temp_parts.append(t)
    rain_parts.append(r)

daily = (obs.merge(pd.concat(temp_parts), on=["city", "date"])
            .merge(pd.concat(rain_parts), on=["city", "date"]))
print("daily table:", daily.shape)

# ---------- Check: which day grouping matches the station rain better? ----------
ok = daily[daily["date"].between(RAIN_START, RAIN_END)]
for city, g in ok.groupby("city"):
    print(f"{city:20s} corr, calendar day: {g['PRCP'].corr(g['fc_rain_calday_lead1']):.3f}"
          f" | 08:30 window: {g['PRCP'].corr(g['fc_rain_lead1']):.3f}")
daily = daily.drop(columns="fc_rain_calday_lead1")

station rows: 3145 -> calendar rows: 3564
daily table: (3564, 19)
Kolkata              corr, calendar day: 0.448 | 08:30 window: 0.461
Thiruvananthapuram   corr, calendar day: 0.449 | 08:30 window: 0.365
Visakhapatnam        corr, calendar day: 0.319 | 08:30 window: 0.542


In [16]:
# ---------- Helper columns and forecast column names ----------
daily["year"] = daily["date"].dt.year
daily["month"] = daily["date"].dt.month
tcols = [f"fc_tmax_lead{k}" for k in LEADS]
rcols = [f"fc_rain_lead{k}" for k in LEADS]

# ---------- Step 1: end of the station record ----------
# Days after the last station value are not random gaps: the record simply stops there.
last = noaa.groupby(["city", "element"])["date"].max().unstack()
print("last station date per city:\n", last[["TMAX", "PRCP"]].to_string())
daily["tmax_in_record"] = daily["date"] <= daily["city"].map(last["TMAX"])
daily["prcp_in_record"] = daily["date"] <= daily["city"].map(last["PRCP"])

n0 = len(daily)
temp_days = daily[daily["tmax_in_record"]].copy()
add_log("TMAX table: cut days after end of station record", n0, len(temp_days),
        "station record stops before the window end")
in_rain_window = daily["date"].between(RAIN_START, RAIN_END)
n0 = int(in_rain_window.sum())
rain_days = daily[in_rain_window & daily["prcp_in_record"]].copy()
add_log("Rain table: cut days after end of station record", n0, len(rain_days),
        "station record stops before the window end")

# ---------- Step 2: forecast gaps (same days in every city, so the API is the cause) ----------
gapT = temp_days[tcols].isna().any(axis=1)
gapR = rain_days[rcols].isna().any(axis=1)
print("\ntemperature days with a missing forecast lead:", gapT.groupby(temp_days["city"]).sum().to_dict())
print("rain days with a missing forecast lead       :", gapR.groupby(rain_days["city"]).sum().to_dict())
print("gap dates, temperature:", str(temp_days.loc[gapT, "date"].min())[:10], "to", str(temp_days.loc[gapT, "date"].max())[:10])
print("gap dates, rain       :", str(rain_days.loc[gapR, "date"].min())[:10], "to", str(rain_days.loc[gapR, "date"].max())[:10])

# ---------- Missing forecast days per lead column (the 3 cities together) ----------
per_lead = pd.DataFrame({"temperature_window": temp_days[tcols].isna().sum().values,
                         "rain_window": rain_days[rcols].isna().sum().values},
                        index=[f"lead{k}" for k in LEADS])
print("\nmissing forecast days per lead column:")
print(per_lead.to_string())

# ---------- The forecast is not imputed: a day with any lead missing is dropped for all 7 leads ----------
n0 = len(temp_days)
temp_days = temp_days[~gapT].copy()
add_log("TMAX table: drop days with any missing forecast lead", n0, len(temp_days),
        "same days kept for all 7 leads, API gap, not imputed")
n0 = len(rain_days)
rain_days = rain_days[~gapR].copy()
add_log("Rain table: drop days with any missing forecast lead", n0, len(rain_days),
        "same days kept for all 7 leads, API gap, not imputed")
print("\ndays left: temperature", len(temp_days), "| rain", len(rain_days))

last station date per city:
 element                  TMAX       PRCP
city                                    
Kolkata            2025-08-24 2025-08-24
Thiruvananthapuram 2025-08-23 2025-08-21
Visakhapatnam      2025-08-24 2025-08-21
logged: TMAX table: cut days after end of station record
logged: Rain table: cut days after end of station record

temperature days with a missing forecast lead: {'Kolkata': 27, 'Thiruvananthapuram': 27, 'Visakhapatnam': 27}
rain days with a missing forecast lead       : {'Kolkata': 7, 'Thiruvananthapuram': 7, 'Visakhapatnam': 7}
gap dates, temperature: 2023-12-30 to 2024-01-25
gap dates, rain       : 2024-01-20 to 2024-01-26

missing forecast days per lead column:
       temperature_window  rain_window
lead1                  63            3
lead2                  66            6
lead3                  69            9
lead4                  72           12
lead5                  75           15
lead6                  78           18
lead7                  

In [17]:
from statsmodels.tools.tools import add_constant
from statsmodels.discrete.discrete_model import Logit

# ---------- Missing count per column (station values) ----------
tmax_miss = temp_days.groupby("city")["TMAX"].agg(days="size", missing=lambda s: s.isna().sum())
tmax_miss["pct"] = (100 * tmax_miss["missing"] / tmax_miss["days"]).round(1)
prcp_miss = rain_days.groupby("city")["PRCP"].agg(days="size", missing=lambda s: s.isna().sum())
prcp_miss["pct"] = (100 * prcp_miss["missing"] / prcp_miss["days"]).round(1)
print("TMAX (temperature window):\n", tmax_miss.to_string())
print("\nPRCP (rain window):\n", prcp_miss.to_string())

# ---------- Pattern: missing % by year (TMAX) and by month (PRCP) ----------
print("\nTMAX missing % by year:")
print(temp_days.groupby(["city", "year"])["TMAX"].apply(lambda s: round(100 * s.isna().mean(), 1)).unstack().to_string())
print("\nPRCP missing % by month:")
print(rain_days.groupby(["city", "month"])["PRCP"].apply(lambda s: round(100 * s.isna().mean(), 1)).unstack().to_string())

# ---------- Pattern: TMAX gap lengths (one-day holes or long streaks?) ----------
gap_t = {}
for city, g in temp_days.sort_values("date").groupby("city"):
    m = g["TMAX"].isna()
    runs = m.groupby((m != m.shift()).cumsum()).sum()
    runs = runs[runs > 0]
    gap_t[city] = {"gaps": len(runs), "median_len": runs.median(), "longest": runs.max()}
print("\nTMAX gap lengths (days):")
print(pd.DataFrame(gap_t).T.to_string())

# ---------- Logistic regression test (course function) ----------
def logistic_regression_test(df, var_with_missing, predictors):
    y = df[var_with_missing].isnull().astype(int)
    X = df[predictors].copy()
    categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
    if categorical_cols:
        X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
    X = X.astype(float)
    X = add_constant(X)
    model = Logit(y, X).fit(disp=0)
    significant_predictors = model.pvalues[model.pvalues < 0.05].index.tolist()
    if 'const' in significant_predictors:
        significant_predictors.remove('const')
    print(model.summary())
    print(f"\nLikelihood-ratio chi2 = {model.llr:.2f} on {int(model.df_model)} df, p = {model.llr_pvalue:.4g}")
    return model, significant_predictors

# ---------- Does PRCP missingness depend on the forecast rain, month, city? ----------
test = pd.get_dummies(rain_days, columns=["city"], drop_first=True)
predictors = ["fc_rain_lead1", "month"] + [c for c in test.columns if c.startswith("city_")]
model, sig = logistic_regression_test(test, "PRCP", predictors)
print("PRCP significant predictors (Wald p < 0.05):", sig)
print("PRCP: MCAR not rejected" if model.llr_pvalue > 0.05 else "PRCP: MCAR rejected")
log.append({"step": "Missing-value test, PRCP", "rows_before": len(test), "rows_after": len(test),
            "rows_affected": 0, "note": f"logistic regression, LR p = {model.llr_pvalue:.4g}"})

# ---------- Does TMAX missingness depend on the forecast TMAX, month, year, city? ----------
test_t = pd.get_dummies(temp_days, columns=["city"], drop_first=True)
pred_t = ["fc_tmax_lead1", "month", "year"] + [c for c in test_t.columns if c.startswith("city_")]
model_t, sig_t = logistic_regression_test(test_t, "TMAX", pred_t)
print("TMAX significant predictors (Wald p < 0.05):", sig_t)
print("TMAX: MCAR not rejected" if model_t.llr_pvalue > 0.05 else "TMAX: MCAR rejected")
log.append({"step": "Missing-value test, TMAX", "rows_before": len(test_t), "rows_after": len(test_t),
            "rows_affected": 0, "note": f"logistic regression, LR p = {model_t.llr_pvalue:.4g}"})

# ---------- Mechanism (argue in the report) ----------
# PRCP: missing days have very low forecast rain, so a missing value mostly means a dry day that was
#       not written down. The value itself drives the gap: MNAR (the test also shows MAR on month and city).
# TMAX: missingness changes with year and city (MAR on time); the test shows if the forecast TMAX also matters.

TMAX (temperature window):
                     days  missing   pct
city                                   
Kolkata             1154      223  19.3
Thiruvananthapuram  1153      327  28.4
Visakhapatnam       1154      123  10.7

PRCP (rain window):
                     days  missing   pct
city                                   
Kolkata              576      332  57.6
Thiruvananthapuram   573      212  37.0
Visakhapatnam        573      356  62.1

TMAX missing % by year:
year                2022  2023  2024  2025
city                                      
Kolkata             11.2  30.3  19.4   9.7
Thiruvananthapuram  13.6  25.9  32.0  40.4
Visakhapatnam        2.8   7.4  16.4  14.4

PRCP missing % by month:
month                 1     2     3     4     5     6     7     8     9     10     11    12
city                                                                                       
Kolkata             97.2  80.7  79.0  88.3  50.0  45.0  21.0  14.5  13.3  35.5  100.0  80.6
Thiruvan

In [18]:
# ---------- Decision: station values are NOT imputed ----------
# The station value is the truth the forecasts are scored against. A filled value would be another
# forecast, and a forecast cannot be tested against another forecast. Missing station days are deleted.

# ---------- TMAX: delete days without a station value ----------
print("TMAX days deleted per city:")
print(temp_days[temp_days["TMAX"].isna()].groupby("city").size().to_string())
n0 = len(temp_days)
temp_clean = temp_days[temp_days["TMAX"].notna()].copy()
add_log("TMAX: delete days without station value", n0, len(temp_clean), "no imputation, station TMAX is the truth")

# ---------- Rain: status of each day in the rain window ----------
rain = rain_days.copy()
rain["obs_status"] = np.where(rain["PRCP"].isna(), "no_value",
                              np.where(rain["PRCP"] == 0, "recorded_zero", "recorded_wet"))

# ---------- Evidence only (not used to fill anything): forecast rain by status, lead 1 ----------
evidence = rain.groupby(["city", "obs_status"])["fc_rain_lead1"].agg(
    days="size", mean_fc_mm=lambda s: s.mean(), pct_fc_ge_1mm=lambda s: 100 * (s >= 1).mean()).round(1)
print("\nforecast rain by what the station recorded:")
print(evidence.to_string())

# ---------- Rain: delete days without a station value ----------
n0 = len(rain)
rain_clean = rain[rain["PRCP"].notna()].copy()
add_log("Rain: delete days without station value", n0, len(rain_clean), "no imputation, station PRCP is the truth")

# ---------- Effect of deleting: recorded days lean wet (state in limitations) ----------
print("\ndays with rain >= 1 mm (%), recorded days only:")
print(rain_clean.groupby("city")["PRCP"].apply(lambda s: round(100 * (s >= 1).mean(), 1)).to_string())

TMAX days deleted per city:
city
Kolkata               223
Thiruvananthapuram    327
Visakhapatnam         123
logged: TMAX: delete days without station value

forecast rain by what the station recorded:
                                  days  mean_fc_mm  pct_fc_ge_1mm
city               obs_status                                    
Kolkata            no_value        332         1.0           12.3
                   recorded_wet    165         8.8           75.8
                   recorded_zero    79         3.6           44.3
Thiruvananthapuram no_value        212         0.6           12.3
                   recorded_wet    249         8.8           62.7
                   recorded_zero   112         2.4           33.9
Visakhapatnam      no_value        356         0.4            8.4
                   recorded_wet    114         5.1           57.0
                   recorded_zero   103         2.2           36.9
logged: Rain: delete days without station value

days with rain >= 1 m

In [19]:
# ---------- Outlier method: modified z-score (MAD), from the course file ----------
def detect_outliers_modified_zscore(data, threshold=3.5):
    median = np.median(data)
    mad = np.median(np.abs(data - median))
    if mad == 0:
        return np.zeros(len(data), dtype=bool)
    modified_z_scores = 0.6745 * np.abs(data - median) / mad
    return modified_z_scores > threshold

# ---------- Neighbour days (for context) ----------
daily = daily.sort_values(["city", "date"]).reset_index(drop=True)
daily["TMAX_prev"] = daily.groupby("city")["TMAX"].shift(1)
daily["TMAX_next"] = daily.groupby("city")["TMAX"].shift(-1)

# ---------- TMAX: anomaly from the median of the same city and month, MAD per city ----------
t = temp_clean.merge(daily[["city", "date", "TMAX_prev", "TMAX_next"]], on=["city", "date"], how="left")
t["median_month"] = t.groupby(["city", "month"])["TMAX"].transform("median")
t["anomaly"] = t["TMAX"] - t["median_month"]
t["outlier"] = False
for city, g in t.groupby("city"):
    t.loc[g.index[detect_outliers_modified_zscore(g["anomaly"].values)], "outlier"] = True

# ---------- TMAX diagnosis: error if impossible or a one-day spike, else real ----------
physical = (t["TMAX"] < 15) | (t["TMAX"] > 50)
up = (t["TMAX"] - t["TMAX_prev"] > 6) & (t["TMAX"] - t["TMAX_next"] > 6)
down = (t["TMAX_prev"] - t["TMAX"] > 6) & (t["TMAX_next"] - t["TMAX"] > 6)
t["diagnosis"] = np.where(t["outlier"], np.where(physical | up | down, "error", "real"), "")
print("TMAX outliers per city:")
print(t[t["outlier"]].groupby(["city", "diagnosis"]).size().unstack(fill_value=0).to_string())
show = t[t["outlier"]].reindex(t[t["outlier"]]["anomaly"].abs().sort_values(ascending=False).index).head(8)
print(show[["city", "date", "TMAX", "median_month", "TMAX_prev", "TMAX_next", "diagnosis"]].to_string(index=False))

# ---------- TMAX action: delete errors, keep real values (no capping) ----------
n0 = len(t)
temp_clean = t[t["diagnosis"] != "error"].drop(columns=["median_month", "anomaly", "outlier", "diagnosis"])
add_log("TMAX outliers: delete errors, keep real", n0, len(temp_clean), "MAD on month anomaly, real values kept")

# ---------- Rain: wet days only, MAD on log(1 + rain) because rain is skewed ----------
wet = rain_clean[rain_clean["PRCP"] > 0].copy()
wet["outlier"] = False
for city, g in wet.groupby("city"):
    wet.loc[g.index[detect_outliers_modified_zscore(np.log1p(g["PRCP"].values))], "outlier"] = True
wet["diagnosis"] = np.where(wet["outlier"], np.where(wet["PRCP"] > 500, "error", "real"), "")
print("\nrain outliers per city:")
print(wet[wet["outlier"]].groupby(["city", "diagnosis"]).size().unstack(fill_value=0).to_string())
print(wet[wet["outlier"]].sort_values("PRCP", ascending=False).head(8)[["city", "date", "PRCP", "fc_rain_lead1", "diagnosis"]].to_string(index=False))

# ---------- Rain action: delete impossible values only, keep heavy rain, no capping ----------
n0 = len(rain_clean)
rain_clean = rain_clean[rain_clean["PRCP"] <= 500].copy()
add_log("Rain: delete impossible rain (> 500 mm/day)", n0, len(rain_clean), "heavy rain kept, not capped")
# Not used to judge: the forecast itself (it would flatter the forecast).


TMAX outliers per city:


diagnosis           error  real
city                           
Kolkata                 1     4
Thiruvananthapuram      0    11
Visakhapatnam           0    11
         city       date  TMAX  median_month  TMAX_prev  TMAX_next diagnosis
      Kolkata 2024-03-20  21.7         33.65       32.0       28.4     error
      Kolkata 2023-04-01  27.5         36.00       29.4        NaN      real
      Kolkata 2022-10-24  24.5         32.80       30.5       36.5      real
Visakhapatnam 2023-06-11  44.6         36.40       44.6       43.6      real
Visakhapatnam 2023-06-10  44.6         36.40       42.0       44.6      real
Visakhapatnam 2023-06-17  44.4         36.40       44.4       41.4      real
Visakhapatnam 2023-06-16  44.4         36.40        NaN       44.4      real
      Kolkata 2024-03-07  26.2         33.65       29.5       31.2      real
logged: TMAX outliers: delete errors, keep real

rain outliers per city:
Empty DataFrame
Columns: []
Index: []
Empty DataFrame
Columns: [city, date

In [20]:
# ---------- Rain: largest values per city (check the top 3) ----------
print(rain_clean.sort_values("PRCP", ascending=False).groupby("city").head(3)
      [["city", "date", "PRCP", "fc_rain_lead1"]].to_string(index=False))

# ---------- Final clean tables ----------
temp_final = temp_clean[["city", "date", "year", "month", "TMAX"] + tcols].reset_index(drop=True)
rain_final = rain_clean[["city", "date", "year", "month", "PRCP", "obs_status"] + rcols].reset_index(drop=True)
print("\ntemperature table:", temp_final.shape, "| rain table:", rain_final.shape)

# ---------- Matched days per city and season (monsoon = June to September) ----------
for name, df in (("temperature", temp_final), ("rain", rain_final)):
    season = np.where(df["month"].isin([6, 7, 8, 9]), "monsoon", "non-monsoon")
    print(f"\nmatched days, {name}:")
    print(df.assign(season=season).groupby(["city", "season"]).size().unstack(fill_value=0).to_string())

# ---------- Cleaning log: steps from the storage notebook and from block 2, then show and save ----------
log[:0] = [
    {"step": "NOAA date: YYYYMMDD number to date", "rows_before": N_NOAA_RAW, "rows_after": N_NOAA_RAW,
     "rows_affected": N_NOAA_RAW, "note": "0 failed (a bad date would raise an error)"},
    {"step": "Open-Meteo time: text to datetime", "rows_before": N_OM_RAW, "rows_after": N_OM_RAW,
     "rows_affected": N_OM_RAW, "note": "0 failed"},
    {"step": "Number strings check", "rows_before": N_NOAA_RAW, "rows_after": N_NOAA_RAW,
     "rows_affected": 0, "note": "NOAA value is int64, forecasts are float64: no numbers stored as text"},
    {"step": "Spelling and case (city, element)", "rows_before": N_NOAA_RAW, "rows_after": N_NOAA_RAW,
     "rows_affected": 0, "note": "0 values changed"},
    {"step": "Units: tenths to deg C and mm", "rows_before": len(noaa), "rows_after": len(noaa),
     "rows_affected": len(noaa), "note": "value / 10, rows converted"},
    {"step": "Station and grid point are the same place", "rows_before": 3, "rows_after": 3,
     "rows_affected": 0, "note": "rule: under 10 km, 3 of 3 matched"},
]
log_df = pd.DataFrame(log).drop_duplicates().reset_index(drop=True)
print()
print(log_df[["step", "rows_before", "rows_after", "rows_affected", "note"]].to_string(index=False))
log_df.to_csv(f"{CLEAN}/cleaning_log_{DATE_COLLECTED}.csv", index=False)

# ---------- Data dictionary: one row per column of each final table ----------
def dict_rows(table, df, window):
    rows = []
    for col in df.columns:
        if col == "city":
            unit, src, allowed = "-", "added by us", "Kolkata, Thiruvananthapuram, Visakhapatnam"
        elif col == "date":
            unit, src, allowed = "date", "calendar rebuilt by us", f"{window[0]} to {window[1]}"
        elif col == "year":
            unit, src, allowed = "year", "derived from date", f"{window[0][:4]} to {window[1][:4]}"
        elif col == "month":
            unit, src, allowed = "month number", "derived from date", "1 to 12"
        elif col == "TMAX":
            unit, src, allowed = "deg C", "NOAA GHCN-Daily (tenths / 10)", "15 to 50"
        elif col == "PRCP":
            unit, src, allowed = "mm per day (08:30 to 08:30 IST)", "NOAA GHCN-Daily (tenths / 10)", "0 to 500"
        elif col == "obs_status":
            unit, src, allowed = "-", "derived from PRCP", "recorded_zero, recorded_wet"
        elif col.startswith("fc_tmax"):
            unit, src, allowed = "deg C", f"Open-Meteo Previous Runs, {MODEL}, highest hour of the day", "15 to 50"
        else:
            unit, src, allowed = ("mm per day (09:00 to 08:00 stamps)",
                                  f"Open-Meteo Previous Runs, {MODEL}, sum of 24 hours", "0 to 500")
        s = df[col]
        has_range = pd.api.types.is_numeric_dtype(s) or pd.api.types.is_datetime64_any_dtype(s)
        rows.append({"table": table, "column": col, "dtype": str(s.dtype), "unit": unit, "source": src,
                     "allowed_range": allowed, "observed_min": s.min() if has_range else "",
                     "observed_max": s.max() if has_range else ""})
    return rows

dictionary = pd.DataFrame(
    dict_rows("temperature", temp_final, (TEMP_START, TEMP_END))
    + dict_rows("rain", rain_final, (RAIN_START, RAIN_END)))
print()
print(dictionary.to_string(index=False))
dictionary.to_csv(f"{CLEAN}/data_dictionary_{DATE_COLLECTED}.csv", index=False)

              city       date  PRCP  fc_rain_lead1
Thiruvananthapuram 2024-05-19 132.1           32.4
Thiruvananthapuram 2024-10-12 112.0           16.8
     Visakhapatnam 2025-08-18 109.0           16.7
           Kolkata 2025-07-08  99.1            8.0
Thiruvananthapuram 2025-05-24  96.0           22.9
           Kolkata 2024-09-26  95.0           49.6
           Kolkata 2024-05-27  90.9           95.2
     Visakhapatnam 2024-09-09  87.1           51.6
     Visakhapatnam 2024-07-19  73.9           43.9

temperature table: (2787, 12) | rain table: (822, 13)

matched days, temperature:
season              monsoon  non-monsoon
city                                    
Kolkata                 401          529
Thiruvananthapuram      356          470
Visakhapatnam           407          624

matched days, rain:
season              monsoon  non-monsoon
city                                    
Kolkata                 155           89
Thiruvananthapuram      183          178
Visakhapatnam    

In [21]:
# ---------- Save the clean tables as Parquet (format: keeps dtypes, small) ----------
tables = {"temperature": temp_final,
          "rain": rain_final}

for name, df in tables.items():
    path = f"{CLEAN}/noaa_openmeteo_{name}_{MODEL}_{DATE_COLLECTED}.parquet"
    df.to_parquet(path, index=False)

    # ---------- Read back and compare shape and dtypes ----------
    back = pd.read_parquet(path)
    cmp = pd.DataFrame({"before": df.dtypes.astype(str), "after": back.dtypes.astype(str)})
    cmp["same"] = cmp["before"] == cmp["after"]
    print(f"\n=== {name}: {path} ({os.path.getsize(path) / 1e3:.0f} KB) ===")
    print("shape:", df.shape, "->", back.shape, "| same:", df.shape == back.shape)
    print("all dtypes same:", bool(cmp["same"].all()))
    if not cmp["same"].all():
        print(cmp[~cmp["same"]].to_string())

# ---------- Files in clean/ ----------
print("\nfiles in clean/:")
for f in sorted(os.listdir(CLEAN)):
    print(" ", f)


=== temperature: clean/noaa_openmeteo_temperature_gfs_seamless_2026-11-05.parquet (49 KB) ===
shape: (2787, 12) -> (2787, 12) | same: True
all dtypes same: True

=== rain: clean/noaa_openmeteo_rain_gfs_seamless_2026-11-05.parquet (25 KB) ===
shape: (822, 13) -> (822, 13) | same: True
all dtypes same: True

files in clean/:
  cleaning_log_2026-11-05.csv
  data_dictionary_2026-11-05.csv
  noaa_openmeteo_rain_gfs_seamless_2026-11-05.parquet
  noaa_openmeteo_temperature_gfs_seamless_2026-11-05.parquet
